# Demo 03 : Async subagents

Run the cells in order and watch.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · async-subagents

We serve the supervisor and researcher together. The parent runs asynchronously on Agent Server, so the subagent uses ASGI transport. We request each lifecycle operation explicitly; DELAY holds the worker before its model call.

In [2]:
import json
from pathlib import Path

In [3]:
config = {"dependencies": ["."], "graphs": {"service": "./graphs.py:service",
          "researcher": "./graphs.py:researcher", "supervisor": "./graphs.py:supervisor"},
          "env": "../../.env"}
Path("langgraph.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))

{
  "dependencies": [
    "."
  ],
  "graphs": {
    "service": "./graphs.py:service",
    "researcher": "./graphs.py:researcher",
    "supervisor": "./graphs.py:supervisor"
  },
  "env": "../../.env"
}


In [4]:
import atexit
import os
import signal
import socket
import subprocess
import time
from pathlib import Path

import httpx
from langgraph_sdk import get_sync_client

HERE = Path.cwd()
servers = []


def port_open(port):
    with socket.socket() as probe:
        probe.settimeout(0.5)
        return probe.connect_ex(("127.0.0.1", port)) == 0


def wait_port_free(port, seconds=60):
    for _ in range(seconds * 2):
        if not port_open(port):
            return
        time.sleep(0.5)
    raise TimeoutError(f"Port {port} stayed busy after stop.")


def stop_server(proc):
    if proc is None:
        return
    port = getattr(proc, "course_port", None)
    was_running = proc.poll() is None
    if was_running:
        proc.send_signal(signal.CTRL_BREAK_EVENT if os.name == "nt" else signal.SIGINT)
        try:
            proc.wait(timeout=25)
        except subprocess.TimeoutExpired:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(proc.pid), "/T", "/F"],
                               capture_output=True, check=False)
            else:
                proc.kill()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
    if was_running and port is not None:
        wait_port_free(port, seconds=90)


def start_server(port):
    if port not in (2134, 2135):
        raise ValueError("Use a module 09 port.")
    wait_port_free(port, seconds=90)
    log = (HERE / f"server-{port}.log").open("w", encoding="utf-8")
    env = dict(os.environ, PYTHONIOENCODING="utf-8", LANGGRAPH_CLI_NO_ANALYTICS="1")
    env["LANGSMITH_TRACING"] = "false"
    proc = subprocess.Popen(
        ["uv", "run", "langgraph", "dev", "--no-browser", "--no-reload", "--port", str(port),
         "--n-jobs-per-worker", "8", "--studio-url", "https://eu.smith.langchain.com"],
        cwd=HERE, env=env, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0)
    proc.course_port = port
    log.close()
    servers.append(proc)
    atexit.register(stop_server, proc)
    for _ in range(180):
        if proc.poll() is not None:
            raise RuntimeError((HERE / f"server-{port}.log").read_text(encoding="utf-8")[-5000:])
        try:
            response = httpx.get(f"http://127.0.0.1:{port}/ok", timeout=1)
            if response.is_success:
                return proc
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    stop_server(proc)
    raise TimeoutError("Agent Server did not become ready.")


def connect(port):
    return get_sync_client(url=f"http://127.0.0.1:{port}", api_key=None, timeout=240)


def clear_dev_storage():
    import shutil
    target = HERE / ".langgraph_api"
    if target.exists():
        shutil.rmtree(target)
        print("cleared :", target.name)

In [5]:
clear_dev_storage()
server = start_server(2135)
client = connect(2135)
tid = client.threads.create()["thread_id"]
used = set()


def supervise(text):
    result = client.runs.wait(tid, "supervisor", input={"messages": [{"role": "user", "content": text}]},
                              config={"recursion_limit": 30}, raise_error=True)
    if "messages" not in result:
        raise RuntimeError("supervisor run finished without messages : " + repr(result)[:800])
    print("response :", result["messages"][-1]["content"])
    print("tasks :", result.get("async_tasks", {}))
    used.update(c["name"] for m in result["messages"] for c in m.get("tool_calls", []) or [])
    return result


out = supervise("Use start_async_task for parts-researcher. Task: DELAY then look up CE-CF-02 stock. "
                "Return the full task ID and stop. Do not check yet.")
task_id = next(iter(out["async_tasks"]))
print("task ID :", task_id)
assert "start_async_task" in used

cleared : .langgraph_api


response : Task ID: `01a17a43-a73e-74e2-9931-36df0c1176f9`
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-a741-7841-82a5-80fa455dc79e', 'status': 'running', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:18Z', 'last_updated_at': '2026-10-26T22:10:18Z'}}
task ID : 01a17a43-a73e-74e2-9931-36df0c1176f9


We fetch current status, then steer the same task with a new message. The installed update tool interrupts the current run and starts a new run on its thread; it does not alter a live model call. On deepagents 0.7.30 the parent also checks its unfinished tasks before each model call and after its answer, and adds a status update as a user message; such an update can give the model extra turns, so the recursion limit is 30.

In [6]:
out = supervise(f"Use check_async_task for {task_id}. Report its fresh status and stop.")
out = supervise(f"Use update_async_task for {task_id}. Message: NO DELAY. Look up CE-DA-03 instead. "
                "Report the update and stop.")
task = out["async_tasks"][task_id]
client.runs.join(task_id, task["run_id"])
out = supervise(f"Use check_async_task for {task_id}. Return the result, stock and depot.")
assert out["async_tasks"][task_id]["status"] == "success"
print("status updates pushed into the parent :", [str(m["content"])[:200] for m in out["messages"]
      if m["type"] == "human" and not str(m["content"]).startswith("Use ")])

response : Fresh status: running.
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-a741-7841-82a5-80fa455dc79e', 'status': 'running', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:21Z', 'last_updated_at': '2026-10-26T22:10:18Z'}}


response : Updated task `01a17a43-a73e-74e2-9931-36df0c1176f9`: NO DELAY; look up CE-DA-03 instead.
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-ce22-7c13-b12e-2ee87bba3fcb', 'status': 'running', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:29Z', 'last_updated_at': '2026-10-26T22:10:29Z'}}


response : CE-DA-03: stock 0 at Rietpoort.
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-ce22-7c13-b12e-2ee87bba3fcb', 'status': 'success', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:36Z', 'last_updated_at': '2026-10-26T22:10:34Z'}}
status updates pushed into the parent : ['[{\'type\': \'text\', \'text\': "[SYSTEM NOTIFICATION - NOT USER INPUT]\\nAn async subagent task you started has an update. This is not a message from the user: don\'t treat it as their answer to any question']


We launch another held task and cancel it before its model call. We list tracked tasks using a fresh server status lookup.

In [7]:
old_ids = set(out["async_tasks"])
out = supervise("Use start_async_task for parts-researcher. Task: DELAY then look up CE-CF-02 stock. "
                "Return its ID and stop.")
cancel_id = next(iter(set(out["async_tasks"]) - old_ids))
out = supervise(f"Use cancel_async_task for {cancel_id}. Return the cancellation and stop.")
out = supervise("Use list_async_tasks with status_filter all. Report the current list and stop.")
print("lifecycle tools :", sorted(used))
assert {"start_async_task", "check_async_task", "update_async_task", "cancel_async_task", "list_async_tasks"} <= used

response : Task ID: `01a17a44-097a-7732-9adb-fbe5baa2eccd`
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-ce22-7c13-b12e-2ee87bba3fcb', 'status': 'success', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:36Z', 'last_updated_at': '2026-10-26T22:10:34Z'}, '01a17a44-097a-7732-9adb-fbe5baa2eccd': {'task_id': '01a17a44-097a-7732-9adb-fbe5baa2eccd', 'agent_name': 'parts-researcher', 'thread_id': '01a17a44-097a-7732-9adb-fbe5baa2eccd', 'run_id': '01a17a44-097d-7183-a181-f7b75d3dd9c4', 'status': 'running', 'created_at': '2026-10-26T22:10:41Z', 'last_checked_at': '2026-10-26T22:10:44Z', 'last_updated_at': '2026-10-26T22:10:44Z'}}


response : Cancelled task `01a17a44-097a-7732-9adb-fbe5baa2eccd`.
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-ce22-7c13-b12e-2ee87bba3fcb', 'status': 'success', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:36Z', 'last_updated_at': '2026-10-26T22:10:34Z'}, '01a17a44-097a-7732-9adb-fbe5baa2eccd': {'task_id': '01a17a44-097a-7732-9adb-fbe5baa2eccd', 'agent_name': 'parts-researcher', 'thread_id': '01a17a44-097a-7732-9adb-fbe5baa2eccd', 'run_id': '01a17a44-097d-7183-a181-f7b75d3dd9c4', 'status': 'cancelled', 'created_at': '2026-10-26T22:10:41Z', 'last_checked_at': '2026-10-26T22:10:47Z', 'last_updated_at': '2026-10-26T22:10:47Z'}}


response : Current tasks:
- `01a17a43-a73e-74e2-9931-36df0c1176f9` — parts-researcher — success
- `01a17a44-097a-7732-9adb-fbe5baa2eccd` — parts-researcher — cancelled
tasks : {'01a17a43-a73e-74e2-9931-36df0c1176f9': {'task_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'agent_name': 'parts-researcher', 'thread_id': '01a17a43-a73e-74e2-9931-36df0c1176f9', 'run_id': '01a17a43-ce22-7c13-b12e-2ee87bba3fcb', 'status': 'success', 'created_at': '2026-10-26T22:10:16Z', 'last_checked_at': '2026-10-26T22:10:51Z', 'last_updated_at': '2026-10-26T22:10:34Z'}, '01a17a44-097a-7732-9adb-fbe5baa2eccd': {'task_id': '01a17a44-097a-7732-9adb-fbe5baa2eccd', 'agent_name': 'parts-researcher', 'thread_id': '01a17a44-097a-7732-9adb-fbe5baa2eccd', 'run_id': '01a17a44-097d-7183-a181-f7b75d3dd9c4', 'status': 'cancelled', 'created_at': '2026-10-26T22:10:41Z', 'last_checked_at': '2026-10-26T22:10:51Z', 'last_updated_at': '2026-10-26T22:10:47Z'}}
lifecycle tools : ['cancel_async_task', 'check_async_task', 'list_async

In [8]:
for proc in servers:
    stop_server(proc)
print("servers stopped :", all(proc.poll() is not None for proc in servers))

servers stopped : True
